# Data Processing

In [52]:
import pandas as pd
import os

In [ ]:
analysis_positions = [1, 3, 5, 7, 9]

manual_path = "/home/vilota/566-qa-2/618D/IMG/MTF数据/结果数据-总检2026-04-29-combine(结果数据 (整理结果数据）).csv"

df = pd.read_csv(manual_path)

df = df.rename(columns={"????": "Capture Time", "??": "Grade", "NG??": "NG", "???": "SN", "??.1": "Short SN Pairing"})
column_to_drop = ['Capture Time', 'Grade', 'NG', 'Short SN Pairing', 'Full SN', '???.1', '???.2', '???_Focus', '???_Focus.1', 'AA?_??S', 'AA?_??S2', 'AA?_??T', 'AA?_??T2', 'AA?_??S.1', 'AA?_??T.1', 'AA?_??S.2', 'AA?_??T.2', 'AA?_??S.3', 'AA?_??T.3', 'AA?_??S.4', 'AA?_??T.4', 'AA?_Tilt-X', 'AA?_Tilt-Y', 'AA?_OC-X', 'AA?_OC-Y', 'AA?_???X', 'AA?_???Y', 'AA?_??0.5F-S', 'AA?_??0.5F-T', 'AA?_??0.5F-S.1', 'AA?_??0.5F-T.1', 'AA?_??0.5F-S.2', 'AA?_??0.5F-T.2', 'AA?_??0.5F-S.3', 'AA?_??0.5F-T.3', 'AA?_????', 'AA?_????.1', 'AA?_????.2', 'AA?_????.3', 'AA?_??0.5??', 'AA?_??0.5??.1', 'AA?_??0.5??.2', 'AA?_??0.5??.3']
df = df.drop(columns=column_to_drop, errors='ignore')
df = df.fillna("o")

pos_columns = ['pos 1', 'pos 3', 'pos 5', 'pos 7', 'pos 9']

for col in pos_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.split('/').str[0].str.strip()

df['SN'] = df['SN'].astype(str).str.split('/').str[0].str.strip().str.upper().str.replace(r'\.0$', '', regex=True)


In [54]:
df.head()

,SN,pos 5,pos 1,pos 3,pos 7,pos 9
0,1359,o,o,sn,sn,mn
1,1365,o,o,o,o,o
2,1384,sf,o,o,o,o
3,1388,o,o,sf,o,o
4,1393,xf,sf,o,o,o


In [55]:
predict_path = "/home/vilota/mingjie/dinov3/scripts/consolidated_batch_predictions.csv"
df2 = pd.read_csv(predict_path)
df2['SN'] = df2['SN'].astype(str).str.split('/').str[0].str.strip().str.upper().str.replace(r'\.0$', '', regex=True)

merged_df = pd.merge(df, df2, on='SN', how='inner')

# Compare result

In [56]:
# 1. Initialize counters for each row at 0
merged_df['number of correct "o"'] = 0
merged_df['number of incorrect "o"'] = 0

# 2. Check each position using clean vectorized comparisons
for pos in [1, 3, 5, 7, 9]:
    human_col = f'pos {pos}'          # Matches 'pos 1' from your human sheet
    pred_col = f'pos {pos} predict'   # Matches 'pos 1 predict' from the model sheet
    
    # Ensure values are lowercase and stripped for clean matching
    h_series = merged_df[human_col].astype(str).str.strip().str.lower()
    p_series = merged_df[pred_col].astype(str).str.strip().str.lower()
    
    # Model said 'o' and Human said 'o'
    merged_df['number of correct "o"'] += (p_series == 'o') & (h_series == 'o')
    
    # Model said 'o' but Human said something else (Defect)
    merged_df['number of incorrect "o"'] += (p_series == 'o') & (h_series != 'o')



In [57]:
total_correct_o = merged_df['number of correct "o"'].sum()
total_incorrect_o = merged_df['number of incorrect "o"'].sum()

# Define the exact clean 13579 column structure
rearranged_columns = [
    'SN',
    # 1. Human annotations grouped (13579)
    'pos 1', 'pos 3', 'pos 5', 'pos 7', 'pos 9',
    # 2. Model predictions and confidence rates grouped (13579)
    'pos 1 predict', 'pos 1 confidence',
    'pos 3 predict', 'pos 3 confidence',
    'pos 5 predict', 'pos 5 confidence',
    'pos 7 predict', 'pos 7 confidence',
    'pos 9 predict', 'pos 9 confidence',
    # 3. Decision metrics and counters
    'Action Required',
    'number of correct "o"',
    'number of incorrect "o"'
]

# Apply the new column sorting order to your merged DataFrame
merged_df = merged_df[rearranged_columns]

# Save the final file
merged_df.to_csv("final_evaluation_results.csv", index=False)

print(f"total number of correct 'o' predictions: {total_correct_o}")
print(f"total number of incorrect 'o' predictions: {total_incorrect_o}")

total number of correct 'o' predictions: 511
total number of incorrect 'o' predictions: 72


# Check Performance for certain baseline (all)

In [58]:
# 1. Specify your baseline threshold here (as a percentage number)
BASELINE_THRESHOLD = 75  # Change this to whatever baseline you want to test

# Ensure we don't include the TOTAL SUM row if it's already there
df_filtered = merged_df[merged_df['SN'] != 'TOTAL SUM'].copy()

total_correct_above = 0
total_incorrect_above = 0
total_dataset_predictions = 0

# 2. Loop through all positions to check predictions against the baseline
for pos in [1, 3, 5, 7, 9]:
    human_col = f'pos {pos}'
    pred_col = f'pos {pos} predict'
    conf_col = f'pos {pos} confidence'
    
    # Clean up columns (handle strings, lowercase, and convert confidence to float)
    h_series = df_filtered[human_col].astype(str).str.strip().str.lower()
    p_series = df_filtered[pred_col].astype(str).str.strip().str.lower()
    conf_series = df_filtered[conf_col].astype(str).str.replace('%', '').astype(float)
    
    # Track the grand total of ALL evaluations across the whole dataset
    total_dataset_predictions += len(df_filtered)
    
    # Create a mask for predictions that are AT or ABOVE your baseline
    above_baseline_mask = conf_series >= BASELINE_THRESHOLD
    
    # Count how many are correct vs incorrect inside this baseline group
    correct_mask = (p_series == h_series) & above_baseline_mask
    incorrect_mask = (p_series != h_series) & above_baseline_mask
    
    total_correct_above += correct_mask.sum()
    total_incorrect_above += incorrect_mask.sum()

total_above = total_correct_above + total_incorrect_above
accuracy_above_baseline = (total_correct_above / total_above * 100) if total_above > 0 else 0

# -----------------------------------------------------------
# 📊 CALCULATE AUTOMATION COVERAGE RATIO
# -----------------------------------------------------------
# Calculate what percentage of all predictions are at or above the baseline
percentage_above_baseline = (total_above / total_dataset_predictions * 100) if total_dataset_predictions > 0 else 0

# 3. Print the results to your screen
print(f"--- 📊 General Prediction Analysis for Baseline >= {BASELINE_THRESHOLD}% ---")
print(f"Total predictions above baseline:      {total_above} out of {total_dataset_predictions} total dataset samples")
print(f"Percentage of predictions above bsln:  {percentage_above_baseline:.2f}% (Workflow coverage)")
print(f"----------------------------------------------------------------------")
print(f"Number of CORRECT predictions:         {total_correct_above}")
print(f"Number of INCORRECT predictions:       {total_incorrect_above}")
print(f"Accuracy above baseline:               {accuracy_above_baseline:.2f}%")

if total_incorrect_above == 0 and (total_correct_above > 0):
    print(f"\n🎉 Success! {BASELINE_THRESHOLD}% is a safe baseline. Content above this is 100% correct.")
else:
    print(f"\n⚠️ Warning: Still found {total_incorrect_above} mistake(s) above {BASELINE_THRESHOLD}%. Raise the threshold.")

--- 📊 General Prediction Analysis for Baseline >= 75% ---
Total predictions above baseline:      609 out of 1515 total dataset samples
Percentage of predictions above bsln:  40.20% (Workflow coverage)
----------------------------------------------------------------------
Number of CORRECT predictions:         518
Number of INCORRECT predictions:       91
Accuracy above baseline:               85.06%

⚠️ Warning: Still found 91 mistake(s) above 75%. Raise the threshold.


# Check Performance for certain baseline ("o")

In [59]:
# 1. Specify your baseline threshold here (as a percentage number)
BASELINE_THRESHOLD = 80  # Change this to whatever baseline you want to test

# Ensure we don't include the TOTAL SUM row if it's already there
df_filtered = merged_df[merged_df['SN'] != 'TOTAL SUM'].copy()

total_correct_o_above = 0
total_incorrect_o_above = 0
total_dataset_predictions = 0

# 2. Loop through all positions to check predictions against the baseline
for pos in [1, 3, 5, 7, 9]:
    human_col = f'pos {pos}'
    pred_col = f'pos {pos} predict'
    conf_col = f'pos {pos} confidence'
    
    # Clean up columns (handle strings, lowercase, and convert confidence to float)
    h_series = df_filtered[human_col].astype(str).str.strip().str.lower()
    p_series = df_filtered[pred_col].astype(str).str.strip().str.lower()
    conf_series = df_filtered[conf_col].astype(str).str.replace('%', '').astype(float)
    
    # Keep track of the grand total of ALL evaluations across the whole dataset
    total_dataset_predictions += len(df_filtered)
    
    # Create masks:
    # - Prediction must be AT or ABOVE your baseline
    # - Prediction MUST specifically be 'o'
    above_baseline_mask = conf_series >= BASELINE_THRESHOLD
    predict_is_o_mask = p_series == 'o'
    
    # Combine conditions: looking strictly for high-confidence 'o' predictions
    target_mask = predict_is_o_mask & above_baseline_mask
    
    # Correct 'o': Model predicted 'o' and Human agreed it was 'o'
    correct_o_mask = (h_series == 'o') & target_mask
    
    # Incorrect 'o' (False OK / Missed Defect): Model predicted 'o' but Human flagged a defect
    incorrect_o_mask = (h_series != 'o') & target_mask
    
    total_correct_o_above += correct_o_mask.sum()
    total_incorrect_o_above += incorrect_o_mask.sum()

total_o_above = total_correct_o_above + total_incorrect_o_above
accuracy_above_baseline = (total_correct_o_above / total_o_above * 100) if total_o_above > 0 else 0

# -----------------------------------------------------------
# 📊 CALCULATE AUTOMATION COVERAGE RATIO
# -----------------------------------------------------------
# Calculate what percentage of the absolute entire dataset is automated by this threshold
percentage_automated = (total_o_above / total_dataset_predictions * 100) if total_dataset_predictions > 0 else 0

# 3. Print the results to your screen
print(f"--- 📊 'o' (OK) Prediction Analysis for Baseline >= {BASELINE_THRESHOLD}% ---")
print(f"Total 'o' predictions above baseline:  {total_o_above} out of {total_dataset_predictions} total dataset samples")
print(f"Automation Coverage Rate:              {percentage_automated:.2f}% (Percentage of total workflow cleared)")
print(f"----------------------------------------------------------------------")
print(f"Number of CORRECT 'o' predictions:     {total_correct_o_above}")
print(f"Number of INCORRECT 'o' (False OKs):   {total_incorrect_o_above}")
print(f"Precision Score above baseline:        {accuracy_above_baseline:.2f}%")

if total_incorrect_o_above == 0 and (total_correct_o_above > 0):
    print(f"\n🎉 Success! At >= {BASELINE_THRESHOLD}%, every single automated 'o' pass is 100% correct.")
else:
    print(f"\n⚠️ Warning: Found {total_incorrect_o_above} missed defect(s) allowed past your baseline! Raise the threshold.")

--- 📊 'o' (OK) Prediction Analysis for Baseline >= 80% ---
Total 'o' predictions above baseline:  583 out of 1515 total dataset samples
Automation Coverage Rate:              38.48% (Percentage of total workflow cleared)
----------------------------------------------------------------------
Number of CORRECT 'o' predictions:     511
Number of INCORRECT 'o' (False OKs):   72
Precision Score above baseline:        87.65%

⚠️ Warning: Found 72 missed defect(s) allowed past your baseline! Raise the threshold.
